In [ ]:
!pip install -q --upgrade transformers accelerate peft trl datasets bitsandbytes

In [ ]:
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q --upgrade peft trl datasets accelerate bitsandbytes
import transformers; print(f"transformers: {transformers.__version__}")

In [ ]:
import transformers; print(f"transformers: {transformers.__version__}")

In [ ]:
import torch
print(f"GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  {torch.cuda.get_device_name(i)}: {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")

In [ ]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

In [1]:
import os
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, TaskType
from trl import SFTConfig, SFTTrainer
 
# --- Verify GPU ---
print(f"GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({props.total_memory / 1e9:.1f} GB)")
 
# --- Config ---
MODEL_ID = "Qwen/Qwen3.5-4B"
DATA_DIR = "/kaggle/input/datasets/yuanmazax/delta-filing"
OUTPUT_DIR = "/kaggle/working/adapters/sft_pytorch"
 
# --- Chat template without thinking tokens ---
SIMPLE_CHAT_TEMPLATE = (
    "{% for message in messages %}"
    "{% if message['role'] == 'system' %}"
    "<|im_start|>system\n{{ message['content'] | trim }}<|im_end|>\n"
    "{% elif message['role'] == 'user' %}"
    "<|im_start|>user\n{{ message['content'] | trim }}<|im_end|>\n"
    "{% elif message['role'] == 'assistant' %}"
    "<|im_start|>assistant\n{{ message['content'] | trim }}<|im_end|>\n"
    "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
    "<|im_start|>assistant\n"
    "{% endif %}"
)
 
# --- Tokenizer ---
print("\n[1] Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.chat_template = SIMPLE_CHAT_TEMPLATE
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
 
# Verify no thinking tokens
test_text = tokenizer.apply_chat_template(
    [{"role": "user", "content": "test"}],
    tokenize=False, add_generation_prompt=True,
)
assert "<think>" not in test_text, f"Thinking tokens found: {test_text}"
print("  Chat template: OK (no thinking tokens)")
 
# --- Load model with 4-bit quantization (QLoRA) ---
print("\n[2] Loading model with 4-bit quantization...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
 
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    trust_remote_code=True,
    device_map="auto",
)
 
# VERIFY quantization is working
memory_gb = model.get_memory_footprint() / 1e9
print(f"  Model memory: {memory_gb:.1f} GB")
assert memory_gb < 5.0, f"Model using {memory_gb:.1f} GB — quantization NOT working!"
print(f"  4-bit quantization: VERIFIED")
 
# --- LoRA config ---
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
)
 
# --- Dataset ---
print("\n[3] Loading dataset...")
dataset = load_dataset(
    "json",
    data_files={
        "train": f"{DATA_DIR}/train.jsonl",
        "valid": f"{DATA_DIR}/valid.jsonl",
    },
)
print(f"  Train: {len(dataset['train'])}, Valid: {len(dataset['valid'])}")
 
# Verify first example
sample = dataset["train"][0]
sample_text = tokenizer.apply_chat_template(sample["messages"], tokenize=False)
sample_tokens = tokenizer.encode(sample_text)
print(f"  Sample tokens: {len(sample_tokens)}")
print(f"  Sample: {sample_text[:150]}...")
 
# --- Training config ---
print("\n[4] Configuring training...")
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=4,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    max_length=2048,
    max_grad_norm=1.0,
    packing=False,
    report_to="none",
    gradient_checkpointing=True,
)
 
# --- Trainer ---
# Pass model OBJECT (not string) to ensure QLoRA is used
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["valid"],
    peft_config=lora_config,
    processing_class=tokenizer,
)
 
# --- Check GPU memory before training ---
for i in range(torch.cuda.device_count()):
    allocated = torch.cuda.memory_allocated(i) / 1e9
    reserved = torch.cuda.memory_reserved(i) / 1e9
    print(f"  GPU {i}: {allocated:.1f} GB allocated, {reserved:.1f} GB reserved")
 
# --- Train ---
print("\n[5] Starting training...")
trainer.train()
 
# --- Save ---
print("\n[6] Saving...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"  Adapter saved to {OUTPUT_DIR}")

GPUs: 2
  GPU 0: Tesla T4 (15.6 GB)
  GPU 1: Tesla T4 (15.6 GB)

[1] Loading tokenizer...
  Chat template: OK (no thinking tokens)

[2] Loading model with 4-bit quantization...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

  Model memory: 3.1 GB
  4-bit quantization: VERIFIED

[3] Loading dataset...
  Train: 862, Valid: 108
  Sample tokens: 399
  Sample: <|im_start|>system
You are Delta Filing, a financial analyst AI specializing in SEC filing analysis. You analyze 10-K and 10-Q filings, detect year-ov...

[4] Configuring training...


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046, 'pad_token_id': 248044}.


  GPU 0: 1.3 GB allocated, 1.3 GB reserved
  GPU 1: 1.8 GB allocated, 1.9 GB reserved

[5] Starting training...


Step,Training Loss,Validation Loss
50,0.931010,0.929909
100,0.583742,0.632947
150,0.683778,0.602903
200,0.648689,0.590701
250,0.558628,0.580921
300,0.713531,0.577501
350,0.577687,0.575463
400,0.532885,0.575098
432,0.558959,0.575137



[6] Saving...
  Adapter saved to /kaggle/working/adapters/sft_pytorch


In [2]:
!zip -r /kaggle/working/sft_adapter.zip /kaggle/working/adapters/sft_pytorch/

  adding: kaggle/working/adapters/sft_pytorch/ (stored 0%)
  adding: kaggle/working/adapters/sft_pytorch/adapter_config.json (deflated 59%)
  adding: kaggle/working/adapters/sft_pytorch/checkpoint-400/ (stored 0%)
  adding: kaggle/working/adapters/sft_pytorch/checkpoint-400/adapter_config.json (deflated 59%)
  adding: kaggle/working/adapters/sft_pytorch/checkpoint-400/scheduler.pt (deflated 61%)
  adding: kaggle/working/adapters/sft_pytorch/checkpoint-400/tokenizer.json (deflated 80%)
  adding: kaggle/working/adapters/sft_pytorch/checkpoint-400/rng_state.pth (deflated 26%)
  adding: kaggle/working/adapters/sft_pytorch/checkpoint-400/README.md (deflated 65%)
  adding: kaggle/working/adapters/sft_pytorch/checkpoint-400/optimizer.pt (deflated 23%)
  adding: kaggle/working/adapters/sft_pytorch/checkpoint-400/tokenizer_config.json (deflated 65%)
  adding: kaggle/working/adapters/sft_pytorch/checkpoint-400/trainer_state.json (deflated 76%)
  adding: kaggle/working/adapters/sft_pytorch/checkp

In [3]:
from IPython.display import FileLink
!zip -r /kaggle/working/sft_adapter.zip /kaggle/working/adapters/sft_pytorch/
FileLink('/kaggle/working/sft_adapter.zip')

updating: kaggle/working/adapters/sft_pytorch/ (stored 0%)
updating: kaggle/working/adapters/sft_pytorch/adapter_config.json (deflated 59%)
updating: kaggle/working/adapters/sft_pytorch/checkpoint-400/ (stored 0%)
updating: kaggle/working/adapters/sft_pytorch/checkpoint-400/adapter_config.json (deflated 59%)
updating: kaggle/working/adapters/sft_pytorch/checkpoint-400/scheduler.pt (deflated 61%)
updating: kaggle/working/adapters/sft_pytorch/checkpoint-400/tokenizer.json (deflated 80%)
updating: kaggle/working/adapters/sft_pytorch/checkpoint-400/rng_state.pth (deflated 26%)
updating: kaggle/working/adapters/sft_pytorch/checkpoint-400/README.md (deflated 65%)
updating: kaggle/working/adapters/sft_pytorch/checkpoint-400/optimizer.pt (deflated 23%)
updating: kaggle/working/adapters/sft_pytorch/checkpoint-400/tokenizer_config.json (deflated 65%)
updating: kaggle/working/adapters/sft_pytorch/checkpoint-400/trainer_state.json (deflated 76%)
updating: kaggle/working/adapters/sft_pytorch/checkp

/kaggle/working/sft_adapter.zip